# NBA Player Performance Prediction

**Predicting 5th-season scoring output and classifying peak performance using machine learning.**

Team: Ben Grigsby, Elliot Weisberg, Evan Carlson, Nolan Glavis, Roy Ho  
Course: STA 141C - Advanced Statistical Computing, UC Davis  
Date: June 2025

---

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import scipy.stats as stats

from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    RandomizedSearchCV, KFold
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
)
from sklearn.linear_model import LinearRegression
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
)
from sklearn.cluster import KMeans
from xgboost import XGBClassifier

sns.set_style('whitegrid')
%matplotlib inline

## 2. Data Cleaning

Filter to players with 5 consecutive post-draft seasons, consolidate per-season stats, update legacy team names.

In [ ]:
data = pd.read_csv('../data/all_seasons.csv')
data['season'] = data['season'].str[:-3]
data = data[~data.isin(['Undrafted']).any(axis=1)]
data['season'] = pd.to_numeric(data['season'])
data['draft_year'] = pd.to_numeric(data['draft_year'])

playerlist = []
for i in range(len(data)):
    if data['season'].iloc[i] - data['draft_year'].iloc[i] == 4:
        playerlist.append(data['player_name'].iloc[i])

name_mask = data['player_name'].isin(playerlist)
diff_mask = (data['season'] - data['draft_year']) <= 4
train = data[name_mask & diff_mask].sort_values(by='player_name')

team_map = {'NJN':'BKN','SEA':'OKC','VAN':'MEM','CHO':'CHA','CHH':'CHA','NOH':'NOP','NOK':'NOP'}
train['team_abbreviation'] = train['team_abbreviation'].replace(team_map)

counts = train['player_name'].value_counts()
counts5 = counts[counts == 5].index
train5 = train[train['player_name'].isin(counts5)]
print(f'Players with 5 consecutive seasons: {train5["player_name"].nunique()}')

In [ ]:
df = train5.copy()
season_counts = df.groupby('player_name')['season'].nunique().sort_values(ascending=False)
players_with_5 = season_counts[season_counts >= 5].index
df_5 = df[df['player_name'].isin(players_with_5)]

df_sorted = df_5.sort_values(by=['player_name','season'])
df_sorted['season_index'] = df_sorted.groupby('player_name').cumcount() + 1

pts_pivot = df_sorted.pivot(index='player_name', columns='season_index', values='pts')
pts_pivot.columns = [f'ptsseason{i}' for i in pts_pivot.columns]

first_season = df_sorted[df_sorted['season_index']==1].set_index('player_name')
features = first_season[['age','team_abbreviation','draft_round','draft_number','player_height','player_weight']]

model_data = pd.concat([features, pts_pivot], axis=1).dropna()
model_data.to_csv('../data/train5final.csv', index=True)
print(f'Final dataset shape: {model_data.shape}')
model_data.head()

## 3. Train-Test Split

In [ ]:
df = pd.read_csv('../data/train5final.csv')
df = df.dropna(subset=['ptsseason5'])

X = df.drop(columns=['player_name','ptsseason5','team_abbreviation','draft_round'])
y = df['ptsseason5']

preprocessor = ColumnTransformer(
    [('cat', OneHotEncoder(handle_unknown='ignore'), [])],
    remainder='passthrough'
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=15)
print(f'Training: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 4. Exploratory Data Analysis

### 4.1 Correlation Heatmap

In [ ]:
numeric_cols = ['age','draft_round','draft_number','player_height','player_weight',
                'ptsseason1','ptsseason2','ptsseason3','ptsseason4','ptsseason5']
plt.figure(figsize=(10,8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

### 4.2 K-Means Clustering

In [ ]:
cluster_features = ['ptsseason1','ptsseason2','ptsseason3','ptsseason4',
                    'age','player_height','player_weight','draft_number']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[cluster_features])

inertias = []
K = range(1,10)
for k in K:
    km = KMeans(n_clusters=k, random_state=15, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.plot(K, inertias, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Distance')
plt.title('Elbow Plot for K-Means')
plt.show()

km = KMeans(n_clusters=3, random_state=15, n_init=10)
df['Cluster'] = km.fit_predict(X_scaled)
print(df.groupby('Cluster')[cluster_features].mean().round(2))

## 5. Regression

### 5.1 OLS

In [ ]:
X_trainC = sm.add_constant(X_train)
X_testC = sm.add_constant(X_test)
OLSmodel = sm.OLS(y_train, X_trainC).fit()
print(OLSmodel.summary())

y_pred = OLSmodel.predict(X_testC)
print(f'\nOLS RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')

residuals = y_test - y_pred
stats.probplot(residuals, dist='norm', plot=plt)
plt.title('Q-Q Plot of Residuals')
plt.show()

plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Points'); plt.ylabel('Predicted Points')
plt.title('Actual vs. Predicted')
plt.show()

### 5.2 Tree-Based Models

In [ ]:
# Decision Tree
tree_model = Pipeline([('preprocess', preprocessor),
                       ('regressor', DecisionTreeRegressor(random_state=15))])
tree_model.fit(X_train, y_train)
y_pred_tree = tree_model.predict(X_test)
print(f'Decision Tree RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tree)):.4f}')
print(f'Decision Tree R2:   {r2_score(y_test, y_pred_tree):.4f}')

# Random Forest
rf_model = Pipeline([('preprocess', preprocessor),
                     ('regressor', RandomForestRegressor(n_estimators=100, random_state=15))])
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print(f'Random Forest RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}')
print(f'Random Forest R2:   {r2_score(y_test, y_pred_rf):.4f}')

# Gradient Boosted Trees
gb_model = Pipeline([('preprocess', preprocessor),
                     ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=15))])
gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)
print(f'Gradient Boosting RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_gb)):.4f}')
print(f'Gradient Boosting R2:   {r2_score(y_test, y_pred_gb):.4f}')

### 5.3 Cross-Validation

In [ ]:
feats = ['ptsseason1','ptsseason2','ptsseason3','ptsseason4','age','player_height','player_weight','draft_number']
X_cv = df[feats]; y_cv = df['ptsseason5']
X_train_cv, _, y_train_cv, _ = train_test_split(X_cv, y_cv, test_size=0.30, random_state=15)
kf = KFold(n_splits=3, shuffle=True, random_state=15)

models = {'Linear Regression': LinearRegression(),
          'Random Forest': RandomForestRegressor(n_estimators=100, random_state=15),
          'Gradient Boosted': GradientBoostingRegressor(n_estimators=100, random_state=15)}

names, rmses = [], []
for name, model in models.items():
    scores = cross_val_score(model, X_train_cv, y_train_cv, cv=kf, scoring='neg_mean_squared_error')
    avg = np.sqrt(-scores.mean())
    names.append(name); rmses.append(avg)
    print(f'{name} - 3-Fold CV RMSE: {avg:.4f}')

plt.bar(names, rmses)
plt.ylabel('Average RMSE')
plt.title('3-Fold CV RMSE')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

## 6. Classification

Predicting whether a player's 5th season is their best.

In [ ]:
data_cls = pd.read_csv('../data/train5final.csv')
data_cls['season5_best'] = data_cls['ptsseason5'] > data_cls[['ptsseason1','ptsseason2','ptsseason3','ptsseason4']].max(axis=1)

X_cls = data_cls.drop(columns=['player_name','team_abbreviation','ptsseason5','season5_best'])
y_cls = data_cls['season5_best']
train_X, test_X, train_y, test_y = train_test_split(X_cls, y_cls, test_size=0.3, random_state=15)

# LDA
lda = LinearDiscriminantAnalysis()
lda.fit(train_X, train_y)
y_proba_lda = lda.predict_proba(test_X)[:,1]
fpr_lda, tpr_lda, _ = roc_curve(test_y, y_proba_lda)
print(f'LDA Accuracy: {accuracy_score(test_y, lda.predict(test_X)):.4f}')

# QDA
qda = QuadraticDiscriminantAnalysis()
qda.fit(train_X, train_y)
y_proba_qda = qda.predict_proba(test_X)[:,1]
fpr_qda, tpr_qda, _ = roc_curve(test_y, y_proba_qda)
print(f'QDA Accuracy: {accuracy_score(test_y, qda.predict(test_X)):.4f}')

# Random Forest
rf_cls = RandomForestClassifier(random_state=15)
rf_cls.fit(train_X, train_y)
y_proba_rf = rf_cls.predict_proba(test_X)[:,1]
fpr_rf, tpr_rf, _ = roc_curve(test_y, y_proba_rf)
print(f'RF Accuracy: {accuracy_score(test_y, (y_proba_rf>.5).astype(int)):.4f}')

# XGBoost
ratio = train_y.value_counts()[False]/train_y.value_counts()[True]
xgb = XGBClassifier(scale_pos_weight=ratio, eval_metric='logloss', random_state=15)
xgb.fit(train_X, train_y)
y_proba_xgb = xgb.predict_proba(test_X)[:,1]
fpr_xgb, tpr_xgb, _ = roc_curve(test_y, y_proba_xgb)
print(f'XGBoost Accuracy: {accuracy_score(test_y, (y_proba_xgb>.5).astype(int)):.4f}')

### ROC Curves

In [ ]:
plt.figure(figsize=(10,7))
for name, fpr, tpr in [('LDA',fpr_lda,tpr_lda), ('QDA',fpr_qda,tpr_qda),
                       ('Random Forest',fpr_rf,tpr_rf), ('XGBoost',fpr_xgb,tpr_xgb)]:
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves'); plt.legend(loc='lower right'); plt.grid(True); plt.show()

## 7. Conclusion

**Regression:** OLS outperformed all tree-based models (RMSE=3.001, R2=0.767). The relationship is fundamentally linear, with Season 4 PPG as the dominant predictor.

**Classification:** Tuned Random Forest achieved the best accuracy at 73.5%. All models struggled with the 2:1 class imbalance.

**Key Takeaway:** Simpler models won for this dataset. The data lacks sufficient nonlinear structure for complex methods to improve on linear regression.